In [ ]:
#| default_exp reports_interactive

In [ ]:
#| export
import panel as pn
import pandas as pd

In [ ]:
#| export
from portfolio.plots import timeseries_plot

In [ ]:
#| export
from bokeh.models.formatters import NumeralTickFormatter

In [ ]:
#| export
from portfolio.portfolio import *

In [ ]:
#| export
pn.extension('tabulator')

In [ ]:
from portfolio.sample_data import *

To create a report you simply pass it one or more portfolios

In [ ]:
rets, rf, cpi = sample_data_se()

In [ ]:
p = Portfolio('60/40', rets, {'bonds': .40, 'stocks': .60}, rf=rf, cpi=cpi)
p

{'bonds': 0.4, 'stocks': 0.6}

## Return Driver Widget

In [ ]:
#| export
def return_drivers_w(p):
    r = p.return_drivers()
    avg = r.mean()
    return pn.Column(timeseries_plot(r - avg, hline=0, title='Rolling Excess Return Mean Centered'),
    pd.DataFrame(avg, columns=['value']).hvplot.bar(title='Avg Excess Return over Period', yformatter=NumeralTickFormatter(format="0.0%"), hover_tooltips=[("Series", "@index"), ("Value", "@value{0.00%}")]))

In [ ]:
#return_drivers_w(p)

## Time Selection Widget

In [ ]:
#| export
def time_period_w():
    years = list(range(1900, 2026))
    months = list(range(1, 13))
    start_year_w = pn.widgets.Select(name='Start Year', options=years, value=1982)
    start_month_w = pn.widgets.Select(name='Start Month', options=months, value=1)
    end_year_w = pn.widgets.Select(name='End Year', options=years, value=2025)
    end_month_w = pn.widgets.Select(name='End Month', options=months, value=1)
    return start_year_w, start_month_w, end_year_w, end_month_w

## Portfolio information Widget

In [ ]:
#| export
def _get_info(p):
    sources = {'cpi': p.full_cpi, 'rets': p.full_rets, 'rf': p.full_rf}
    df = pd.DataFrame({k: {'datastream start': df.index.min(), 'datastream end': df.index.max(), 'sz': len(df)} for k, df in sources.items()}).T
    return pn.widgets.Tabulator(df)

In [ ]:
#_get_info(p)

In [ ]:
#| export
def _allocation_w(p):
    w = pd.DataFrame(index=['Allocation'], data=[p.weights]) * 100
    return pn.widgets.Tabulator(
        w,
        formatters={n: {'type': 'money', 'symbol': '%', 'symbolAfter': True, 'precision': 1} for n in w.columns}
    )

In [ ]:
#_allocation_w(p)

In [ ]:
#| export
def portfolio_info_w(p):
    return pn.Tabs(
        ('Data Streams', _get_info(p)),
        ('Allocation', _allocation_w(p))
    )

In [ ]:
#portfolio_info_w(p)

## Decade by decade widget

In [ ]:
#| export
def decade_w(p):
    return pn.Column(
        pn.pane.Markdown('### Real 10y Return (ann)'), timeseries_plot(p.lost_decade(components=True), hline=0.0, ylabel='Return (ann)'),
        pn.pane.Markdown('### Decade by Decade real return'), timeseries_plot(p.r_by_decade(), logy=True, hline=1.0, xlabel='Years Held', ylabel='Return'),
    )

In [ ]:
#| export
def decade_comparison_w(*portfolios):
    p = portfolios[0]
    decades = list(p.r_by_decade().columns)
    decade_w = pn.widgets.Select(name='Decade', options=decades, value=decades[0])
    def plot(decade):
        df = pd.concat({p.name: p.r_by_decade()[decade] for p in portfolios}, axis=1)
        return timeseries_plot(df, logy=True, is_perc=False, xlabel='Years Held', ylabel='Real Return (cumulative)', hline=1.0)
    return pn.Column(decade_w, pn.bind(plot, decade_w))

In [ ]:
#| export
def lost_decade_comparison_w(*portfolios):
    df = pd.concat({p.name: p.lost_decade() for p in portfolios}, axis=1)
    return timeseries_plot(df, hline=0, ylabel='Real 10y Return (ann)')

## Portfolio Overview Widget

In [ ]:
#| export
def portfolio_overview(p):
    asset_stats   = pn.bind(lambda p: pn.panel(p.asset_stats()), p)
    risk_contrib  = pn.bind(lambda p: pn.panel(p.risk_contribution()), p)
    roll_vol_plot = pn.bind(lambda p: timeseries_plot(p.rolling_vol()), p)
    rc_plot       = pn.bind(lambda p: timeseries_plot(p.rc_simple()), p)
    roll_no_exc   = pn.bind(lambda p: timeseries_plot(p.roll_return(excess=False, extras=True), hline=0), p)
    correlation   = pn.bind(lambda p: pn.panel(p.correlation()), p)
    info = pn.bind(lambda p: portfolio_info_w(p), p)

    decade_analysis = pn.bind(lambda p: decade_w(p), p)

    overview = pn.Column(
        pn.pane.Markdown('### Asset Stats'), asset_stats,
        pn.pane.Markdown('## Risk'),
        pn.pane.Markdown('### Risk Contribution'), risk_contrib,
        pn.pane.Markdown('### Rolling Volatility'), roll_vol_plot,
        pn.pane.Markdown('### Risk Contribution (Plot)'), rc_plot,
        pn.pane.Markdown('### Extra 12m Rolling (No Excess)'), roll_no_exc,
        pn.pane.Markdown('### Drawdowns Assets + Portfolio'), pn.bind(lambda p: timeseries_plot(p.drawdown_series(assets=True), hline=0), p),
    )
    return_drivers = pn.Column(pn.pane.Markdown('### Return Drivers 5 years (excess mean centered'), pn.bind(lambda p: return_drivers_w(p), p))
    corr = pn.Column(pn.pane.Markdown('### Correlation Matrix'), correlation)
    
    return pn.Tabs(
        ('Portfolio Overview', overview),
        ('Correlation Matrix', corr),
        ('Return Drivers', return_drivers),
        ('Info', info),
        ('Decade analysis', decade_analysis),
        )

## Full report

In [ ]:
#| export
def interactive_report(*portfolios):
    single = len(portfolios) == 1
    window_w = pn.widgets.Select(name='Rolling Window (Years)', options=list(range(1,30)), value=1)
    time_w = time_period_w()

    slice_ports = lambda sy, sm, ey, em: [p.between(f'{sm}/{sy}', f'{em}/{ey}') for p in portfolios]
    ports_rx = pn.bind(slice_ports, *time_w)

    summary       = pn.bind(lambda ps: pn.panel(compare(*ps)), ports_rx)
    cum_ret_plot  = pn.bind(lambda ps: timeseries_plot(compare(*ps, metric='cum_excess_return')), ports_rx)
    roll_ret_plot = pn.bind(lambda ps, w: timeseries_plot(compare(*ps, metric='roll_return', months=w*12), interactive_hlines=True), ports_rx, window_w)
    real_w_plot   = pn.bind(lambda ps: timeseries_plot(compare(*ps, metric='real_w'), logy=True, is_perc=False, interactive_growth_lines=True,), ports_rx)
    drawdown_plot = pn.bind(lambda ps: timeseries_plot(compare(*ps, metric='drawdown_series')), ports_rx)

    # decade comparison
    decade_comp = pn.bind(lambda ps: decade_comparison_w(*ps), ports_rx)
    lost_dec_comp = pn.bind(lambda ps: lost_decade_comparison_w(*ps), ports_rx)

    name_w = pn.widgets.Select(name='Portfolio', options=[p.name for p in portfolios])
    selected_p = pn.bind(lambda ps, name: next(p for p in ps if p.name == name), ports_rx, name_w)
    portfolio_overview_w = pn.Column(
        name_w,
        portfolio_overview(selected_p)
    )
    comparison_w = pn.Column(
        pn.pane.Markdown('## Time Period'), pn.Row(*time_w),
        pn.pane.Markdown('## Portfolio Stats'),
        pn.pane.Markdown('### Summary'), summary,
        pn.pane.Markdown('## Portfolio Returns'),
        pn.pane.Markdown('### Cumulative Excess Return'), cum_ret_plot,
        pn.pane.Markdown('### Rolling Excess Return'), window_w, roll_ret_plot,
        pn.pane.Markdown('### Real Wealth (Total Real Cum. Compounded Return)'), real_w_plot,
        pn.pane.Markdown('## Drawdowns'), drawdown_plot,

        pn.pane.Markdown('## Real Return comparison'), lost_dec_comp,
        pn.pane.Markdown('## Decade Comparison'), decade_comp,
        )


    return pn.Tabs(('Portfolio Comparison', comparison_w), ('Portfolio Overview', portfolio_overview_w))

In [ ]:
#pn.panel(interactive_report(p))

In [ ]:
#| eval: false
#pn.panel(interactive_report(p)).save('report.html')